# Lee, Matthews, Lemieux et al. (2025) on dandiset 001195

Paper: Lee, Matthews, Lemieux et al., "Separable Dorsal Raphe Dopamine Projections mediate the Facets
of Loneliness-like state", eLife (2025), doi:10.7554/eLife.105955. The claims file
(`claims/001195/10.7554-elife.105955.yaml`) was written from the eLife reviewed preprint, version 2
(8 July 2025), with its Methods, legends and figure supplements.

Dataset: [DANDI:001195](https://dandiarchive.org/dandiset/001195/0.250408.1733), version
0.250408.1733 (doi:10.48324/dandi.001195/0.250408.1733). It is the only published version, and the
draft has the same 230 assets and size, so it is pinned. The paper cites no version.

The dandiset holds 148 ex vivo patch-clamp cells (one NWB file each) and 41 in vivo imaging
sessions of 14 mice (CNMF-E traces, footprints, the raw one-photon movie and the behaviour video as
an external mp4). It holds no pose estimates, cup interactions, position or annotation of which cup
holds the juvenile, and none of the behavioural, tracing or RNAscope data of Figures 1 to 5 and 8.

| Claim | Tested here |
|---|---|
| fig6df-epsp-ipsp-at-rest | tested |
| fig6gi-firing-responses | tested |
| fig6j-epsp-amplitude-region | tested |
| fig6k-ipsp-amplitude-region | tested |
| fig6-s1de-monosynaptic | blocked: the TTX/4AP sweeps are not marked in the files |
| fig6no-cea-clusters | tested |
| fig6-s1h-cea-cluster-properties | tested |
| fig6pq-blp-clusters | tested |
| fig6-s1i-blp-cluster-properties | tested |
| fig6-s1jk-cluster-voltage-area | tested |
| fig6-s1ln-combined-clusters | tested |
| fig7-s1ef-red-vs-blue | tested |
| fig7-s1hi-red-during-blue | tested |
| fig7e, fig7gh, fig7j, fig7i, fig7m, fig7-s1no, fig7-s1l, fig7-s2 (in vivo imaging) | blocked: no pose, cup interactions or cup sides in the dataset; cell counts are shown in the data check |
| Figures 1 to 5 and 8 (17 claims: tracing, behaviour, RNAscope) | blocked: not in the dataset |

Loading and measures are in `drn_ephys.py`, which says where each parameter comes from. The
notebook streams every file; it takes about 30 minutes, most of it reading the patch-clamp sweeps.

In [ ]:
import collections, json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py, remfile, fsspec, av
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from concurrent.futures import ThreadPoolExecutor
import drn_ephys as de

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 200)
print(de.DANDISET_ID, de.DANDISET_VERSION)

## Loading

The file list of the pinned version, the metadata of every imaging file and the length of every behaviour video, and every patch-clamp cell, reduced to the sweeps the analysis uses as it loads.

In [ ]:
ASSETS = de.assets()
icephys = [a for a in ASSETS if a[0].endswith("_icephys.nwb")]
ophys = [a for a in ASSETS if a[0].endswith("image+ophys.nwb")]
client_assets = {a.path: a for a in de.DandiAPIClient().get_dandiset(de.DANDISET_ID, de.DANDISET_VERSION).get_assets()}
videos = {p: client_assets[p].get_content_url(follow_redirects=1, strip_query=True)
          for p in client_assets if p.endswith(".mp4")}
print(len(ASSETS), "NWB files:", len(icephys), "patch-clamp,", len(ophys), "imaging;", len(videos), "videos")

def ophys_meta(path_url):
    path, url = path_url
    with h5py.File(remfile.File(url), "r") as f:
        r = f["processing/ophys/Fluorescence/RoiResponseSeries"]
        g = f["general"]
        return {
            "path": path, "subject": g["subject/subject_id"][()].decode(),
            "subject_description": g["subject/description"][()].decode(),
            "session": f["session_description"][()].decode(),
            "n_rois": r["data"].shape[1], "n_frames": r["data"].shape[0],
            "rate_hz": float(r["starting_time"].attrs["rate"]),
            "video": f["acquisition/BehaviorVideo/external_file"][0].decode(),
            "virus": g["virus"][()].decode(), "stimulus": g["stimulus"][()].decode(),
            "experiment_description": g["experiment_description"][()].decode(),
            "processing_entries": sorted(f["processing/ophys"].keys()),
            "acquisition_entries": sorted(f["acquisition"].keys()),
        }

def video_meta(path):
    with fsspec.open(videos[path], "rb", block_size=2**20) as fh:
        c = av.open(fh)
        s = c.streams.video[0]
        return {"video": path.split("/", 1)[1], "video_s": c.duration / 1e6, "video_frames": s.frames,
                "video_fps": float(s.average_rate)}

with ThreadPoolExecutor(12) as ex:
    OPHYS = pd.DataFrame(list(ex.map(ophys_meta, ophys)))
    VID = pd.DataFrame(list(ex.map(video_meta, sorted(videos))))
OPHYS = OPHYS.merge(VID, on="video", how="left")
print("imaging metadata read")

In [ ]:
CELLS = de.load_cells(icephys, workers=24)
print(len(CELLS), "cells loaded")

## Data check

In [ ]:
# --- patch clamp: what was loaded
cells = pd.DataFrame([{
    "path": c["path"], "subject": c["subject"], "sex": c["sex"], "cell": c["cell_id"], "region": c["region"],
    "targeted": c["targeted"], "opsin": "ChR2" if "ChR2" in c["virus"] else ("ChrimsonR" if "Chrimson" in c["virus"] else c["virus"]),
    **{f"n_{k}": len(v) for k, v in c["sweeps"].items()}, "dropped_sweeps": c["dropped"],
    "rates": sorted(c["rates"]),
} for c in CELLS]).fillna(0)
for col in ["n_rest", "n_firing", "n_steps", "n_seal", "n_red", "n_blue03", "n_blue02", "n_red_contblue"]:
    if col not in cells: cells[col] = 0
print("cells by opsin and region (region from DandiIcephysMetadata/inferred_layer):")
print(pd.crosstab(cells.opsin, cells.region, margins=True))
print("\ncells by sex (general/subject/sex), opsin and region; the Methods use male mice only:")
print(pd.crosstab([cells.sex, cells.opsin], cells.region, margins=True))
print("female subjects:", sorted(cells[cells.sex != "M"].subject.unique()))
print("\ninferred_layer equals targeted_layer in", (cells.region == cells.targeted).sum(), "of", len(cells), "cells")
print("cell name letter by region:")
print(pd.crosstab(cells.cell.str[0], cells.region))
print("\nsampling rates of the sweeps used (Hz):", sorted({r for rs in cells.rates for r in rs}))
print("subjects:", cells.subject.nunique(), sorted(cells.subject.unique()))
print("\ncells with each kind of sweep, by region (ChR2):")
chr2 = cells[cells.opsin == "ChR2"]
print(pd.DataFrame({k: (chr2[f"n_{k}"] > 0).groupby(chr2.region).sum() for k in ["rest", "firing", "steps", "seal"]}))
print("\nChrimsonR cells with each light condition:")
chrim = cells[cells.opsin == "ChrimsonR"]
print({k: int((chrim[f"n_{k}"] > 0).sum()) for k in ["red", "blue03", "blue02", "red_contblue", "steps"]})
print("\nsweeps not used (other protocols, voltage clamp), total:", int(cells.dropped_sweeps.sum()))

# every text field of the files that could name a drug: the sweep (ch-0 series) descriptions, the
# session description, the stimulus_type column of the sequential recordings table and the
# electrode descriptions; general/ holds no notes, protocol, slices or pharmacology field
descs = collections.Counter(d for c in CELLS for d in c["descriptions"])
texts = {"series descriptions": descs,
         "session_description": collections.Counter(c["session_description"] for c in CELLS),
         "sequential_recordings stimulus_type": collections.Counter(t for c in CELLS for t in c["stimulus_types"]),
         "electrode descriptions": collections.Counter(d for c in CELLS for d in c["electrode_descriptions"])}
print()
for name, cnt in texts.items():
    hits = [d for d in cnt if any(k in d.upper() for k in ("TTX", "4AP", "4-AP", "TETRODOTOXIN", "AMINOPYRIDINE"))]
    print(f"{name}: {len(cnt)} distinct values; naming TTX, 4AP or tetrodotoxin: {hits}")
print("session_description values:", dict(texts["session_description"]))
print("electrode descriptions:", dict(texts["electrode descriptions"]))

# duplicates: identical first 5000 samples of any sweep in two files
owners = collections.defaultdict(set)
for c in CELLS:
    for h in c["hashes"]:
        owners[h].add(c["path"])
dups = [sorted(v) for v in owners.values() if len(v) > 1]
print("\nsweeps found in more than one file:", len(dups))

# subject ids that differ only by a suffix ('4484' and '4484?', '235' and '235 cage', 'in_vivo_5' and
# 'in_vivo_5_2') are probably the same mouse; the files do not say
norm = cells.subject.str.replace(r"(\?| cage|_2)$", "", regex=True)
print("subject ids:", cells.subject.nunique(), "; with the suffixes '?', ' cage' and '_2' removed:", norm.nunique(),
      {k: sorted(v) for k, v in cells.groupby(norm).subject.unique().items() if len(v) > 1})

In [ ]:
# --- patch clamp: units and ranges of what the analysis uses
def summary(x):
    x = np.asarray(x, float)
    return f"min {np.nanmin(x):.2f}, median {np.nanmedian(x):.2f}, max {np.nanmax(x):.2f} (n={np.isfinite(x).sum()})"

# per light sweep: the holding current its description names, the current measured on ch-1 in the
# first 0.2 s, and the action potentials (0 mV crossings) in the 5 s before and the 5.5 s after the light
i0 = int(de.LIGHT_ONSET_S * de.FS)
sw = []
for c in CELLS:
    if "ChR2" not in c["virus"]:
        continue
    for kind in ("rest", "firing"):
        for s in c["sweeps"].get(kind, []):
            m = re.search(r"([+-]?\d+)\s*p[aA]", s["desc"][len(de.LIGHT_PROTOCOL):])
            sw.append({"path": c["path"], "cell": c["cell_id"], "region": c["region"], "kind": kind,
                       "described_pA": int(m.group(1)) if m else 0, "measured_pA": s["hold_pA"],
                       "spikes_before": len(de.spike_times(s["v"][i0 - 50_000:i0])),
                       "spikes_after": len(de.spike_times(s["v"][i0:i0 + 55_000]))})
SW = pd.DataFrame(sw)
ok_i = SW.measured_pA.abs() < 1e4
print("ChR2 light sweeps: measured ch-1 current (pA) by kind, sweeps with ch-1 within +-10 nA:")
print(SW[ok_i].groupby("kind").measured_pA.describe().round(1))
print("\nrest sweeps whose measured current exceeds +20 pA:", int(((SW.kind == "rest") & ok_i & (SW.measured_pA > 20)).sum()),
      "in cells", sorted(SW[(SW.kind == "rest") & ok_i & (SW.measured_pA > 20)].cell.unique()))
print("firing sweeps whose measured current is below +5 pA:", int(((SW.kind == "firing") & (SW.measured_pA < 5)).sum()),
      "in cells", sorted(SW[(SW.kind == "firing") & (SW.measured_pA < 5)].cell.unique()))
r = SW[SW.kind == "rest"]
print(f"\nrest sweeps with an action potential in the 5 s before the light: {(r.spikes_before > 0).sum()} of {len(r)}, "
      f"in {r[r.spikes_before > 0].path.nunique()} cells; these are not subthreshold and are left out of the rest averages")
print("per cell, rest sweeps with spikes before the light / all rest sweeps, and the measured current range (pA):")
pc = r.groupby(["region", "cell"]).agg(spiking=("spikes_before", lambda x: int((x > 0).sum())), n=("spikes_before", "size"),
                                      min_pA=("measured_pA", "min"), max_pA=("measured_pA", "max"))
print(pc[pc.spiking > 0].round(0).to_string())
f_ = SW[SW.kind == "firing"].groupby(["region", "cell"]).agg(n=("spikes_before", "size"), spikes_before=("spikes_before", "sum"),
                                                         measured_min=("measured_pA", "min"), measured_max=("measured_pA", "max"))
print("\nfiring sweeps per cell: spikes in the 5 s before the light, and the measured current (pA):")
print(f_.round(0).to_string())

# failed sweeps: ch-1 outside +-10 nA; their membrane potential is flat at 0 mV
bad = SW[~ok_i]
print(f"\nlight sweeps with ch-1 outside +-10 nA (failed, left out): {len(bad)} in cells {sorted(bad.cell.unique())}; "
      f"their largest |Vm|: {max(np.abs(s['v']).max() for c in CELLS for s in c['sweeps'].get('rest', []) if abs(s['hold_pA']) >= 1e4):.1e} mV")
p10 = SW[(SW.cell == "P10") & (SW.kind == "rest")]
print(f"P10: {len(p10)} rest sweeps, {(p10.measured_pA.abs() >= 1e4).sum()} failed, {(p10.measured_pA.abs() < 1e4).sum()} valid, "
      f"{((p10.measured_pA.abs() < 1e4) & (p10.spikes_before > 0)).sum()} of the valid ones fire before the light; "
      f"{(SW.cell == 'P10').sum() - len(p10)} firing sweeps")
rest_base = {c["cell_id"]: np.median(de.average(de.valid(c["sweeps"]["rest"]))[:40000]) for c in CELLS if "rest" in c["sweeps"]}
print("baseline membrane potential of the valid rest sweeps, per cell (mV):", summary(list(rest_base.values())),
      "; above -45 mV:", {k: round(v, 1) for k, v in rest_base.items() if v > -45})
steps_i = [np.median(s["i"][2000:11000]) - np.median(s["i"][:500]) for c in CELLS for s in c["sweeps"].get("steps", [])]
print("current steps read from ch-1 (pA):", summary(steps_i))
caps = [de.cell_capacitance_pf(c["sweeps"]["seal"]) for c in CELLS if "seal" in c["sweeps"]]
print("cells whose every Q-method fit failed (0 pF or less):", int(np.isnan(caps).sum()), "of", len(caps))
seal_dv = [np.ptp(np.percentile(c["sweeps"]["seal"][0]["v"], [10, 90])) for c in CELLS if "seal" in c["sweeps"]]
print("seal test voltage step (ch-1, mV, 10th to 90th percentile):", summary(seal_dv))
print("capacitance from the seal test (pF):", summary(caps))

# The Methods use male mice only, but the cells of the mice labelled female are needed to reach the paper's
# clustering counts (26 CeA and 27 BLP cells): the cells with current steps, by sex
print("\nChR2 cells with current steps, by sex and region (the paper clusters 26 CeA and 27 BLP cells):")
print(pd.crosstab(chr2[chr2.n_steps > 0].sex, chr2[chr2.n_steps > 0].region, margins=True))
# every cell is used, whatever the sex its file gives; male mice only is a variant at the end

The files do not record when the light was on: the stimulus series of every light sweep are zero.
The helper takes the first pulse 5.468 s into each 30 s light sweep, where the average response
of the responding cells starts. The check below averages the rest sweeps of every ChR2 cell and
compares the deflection at that time with the same measure 6 s later, after the 5.5 s response
window, where no light is given.

In [ ]:
def clock_check(kind="rest", opsin="ChR2"):
    rows = []
    for c in CELLS:
        if kind not in c["sweeps"] or opsin not in c["virus"]:
            continue
        avg = de.average(de.valid(c["sweeps"][kind]))
        base, resp = de.light_window(avg, de.LIGHT_ONSET_S)
        early = resp[: int(0.2 * de.FS)] - base.mean()
        base2, resp2 = de.light_window(avg, de.LIGHT_ONSET_S + 6)
        late = resp2[: int(0.2 * de.FS)] - base2.mean()
        rows.append({"true": np.abs(early).max() / base.std(), "shifted": np.abs(late).max() / base2.std()})
    return pd.DataFrame(rows)

def grand_onset(kind="rest", opsin="ChR2", either_sign=False):
    """Grand average of the normalized EPSPs (or responses of either sign) above 10 SD in 5.3-5.8 s, and the time it
    crosses 10% of its peak."""
    segs = []
    for c in CELLS:
        if kind in c["sweeps"] and opsin in c["virus"]:
            avg = de.average(de.valid(c["sweeps"][kind]))
            base, _ = de.light_window(avg)
            seg = avg[int(5.3 * de.FS):int(5.8 * de.FS)] - base.mean()
            if either_sign and seg.max() < -seg.min():
                seg = -seg
            if seg.max() > 10 * base.std():
                segs.append(seg / seg.max())
    if not segs:
        return 0, None, np.nan
    grand = np.mean(segs, axis=0)
    t = 5.3 + np.arange(len(grand)) / de.FS
    return len(segs), (t, grand), t[np.argmax(grand > 0.1)]

cc = clock_check()
print("max |deflection| in the first 200 ms, in baseline SDs, ChR2 rest averages (n=%d)" % len(cc))
print(" at the assumed onset: median %.1f, cells above 10 SD: %d" % (cc["true"].median(), (cc["true"] > 10).sum()))
print(" 6 s later:           median %.1f, cells above 10 SD: %d" % (cc.shifted.median(), (cc.shifted > 10).sum()))

# onset of the grand average of responding cells
from scipy.signal import find_peaks
n, (t, grand), onset = grand_onset()
print(f"grand average of {n} cells with an EPSP above 10 SD crosses 10% of its peak at {onset:.4f} s")
dg = np.gradient(np.convolve(grand, np.ones(10) / 10, mode="same"))
pk, _ = find_peaks(dg, distance=int(0.02 * de.FS), prominence=dg.max() * 0.05)
pk = t[pk][t[pk] >= onset - 0.005][:8]
print("maxima of its rate of rise from the onset on (s):", np.round(pk, 4), "; intervals (ms):", np.round(np.diff(pk) * 1e3, 1))
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(t, grand); ax.axvline(de.LIGHT_ONSET_S, color="r", lw=.5)
for i in range(8): ax.axvline(de.LIGHT_ONSET_S + i / 30, color="r", lw=.3, ls=":")
ax.set_xlabel("time in sweep (s)"); ax.set_ylabel("normalized Vm"); ax.set_title("grand average, red lines: assumed pulses")
plt.show()

In [ ]:
# --- the light time in the ChrimsonR sweeps (red alone, the two blue conditions, red during constant blue)
rows = []
for kind in ("red", "blue03", "blue02", "red_contblue"):
    cc = clock_check(kind, "Chrimson")
    n, _, onset = grand_onset(kind, "Chrimson", either_sign=True)
    rows.append({"condition": kind, "cells": len(cc), "median SD at onset": cc["true"].median(),
                 "cells >10 SD at onset": int((cc["true"] > 10).sum()), "median SD 6 s later": cc.shifted.median(),
                 "cells >10 SD 6 s later": int((cc.shifted > 10).sum()), "cells in grand average": n,
                 "grand average crosses 10% (s)": onset})
print(pd.DataFrame(rows).round(4).to_string())
print("assumed onset:", de.LIGHT_ONSET_S, "s")

In [ ]:
# --- imaging: what is there, against the paper's counts
OPHYS["condition"] = np.select(
    [OPHYS.session.str.contains("Stim ON"), OPHYS.session.str.startswith("Group housed"), OPHYS.session.str.startswith("24hr")],
    ["GH on", "GH off", "SI off"], "?")
OPHYS["group"] = np.where(OPHYS.subject_description.str.contains("chrimson"), "ChrimsonR", "TdTomato")
print(OPHYS.groupby(["group", "condition"]).agg(mice=("subject", "nunique"), sessions=("path", "size"), rois=("n_rois", "sum")))
print("\nROIs per mouse and condition:")
print(OPHYS.pivot_table(index=["group", "subject"], columns="condition", values="n_rois", aggfunc="sum"))
print("\npaper: ChrimsonR 429, 441 and 484 cells (GH off, GH on, SI off); TdTomato 39, 61 and 76")
print("\nimaging frames:", summary(OPHYS.n_frames), "; stated rate:", OPHYS.rate_hz.unique(), "Hz")
print("behaviour video length (s):", summary(OPHYS.video_s), "; video fps:", OPHYS.video_fps.unique())
OPHYS["imaging_s_at_stated_rate"] = OPHYS.n_frames / OPHYS.rate_hz
OPHYS["imaging_s_at_20hz"] = OPHYS.n_frames / 20.0
print("imaging length at the stated 25 Hz (s):", summary(OPHYS.imaging_s_at_stated_rate))
print("imaging length at the Methods' 20 frames/s (s):", summary(OPHYS.imaging_s_at_20hz))
print("video minus imaging at 20 Hz (s):", summary(OPHYS.video_s - OPHYS.imaging_s_at_20hz))
off = OPHYS[(OPHYS.video_s - OPHYS.imaging_s_at_20hz).abs() > 5]
print("sessions where they differ by more than 5 s:", off[["subject", "condition", "video_s", "imaging_s_at_20hz"]].round(1).to_dict("records"))
print("\nprocessing/ophys holds:", OPHYS.processing_entries.map(tuple).unique())
print("acquisition holds:", OPHYS.acquisition_entries.map(tuple).unique())
for col in ["virus", "stimulus", "experiment_description"]:
    print(f"{col}:", OPHYS[col].unique())

In [ ]:
# --- one frame of each behaviour video, 300 s in: can the cups and the juvenile's cup be seen?
def frame_at(path, t_s=300):
    with fsspec.open(videos[path], "rb", block_size=2**20) as fh:
        c = av.open(fh)
        s = c.streams.video[0]
        c.seek(int(t_s / s.time_base), stream=s)
        for fr in c.decode(s):
            return fr.to_ndarray(format="gray")[::3, ::3]
vid_rows = OPHYS.sort_values(["group", "subject", "condition"]).reset_index(drop=True)
with ThreadPoolExecutor(12) as ex:
    frames = list(ex.map(frame_at, [p.split("/")[0] + "/" + v for p, v in zip(vid_rows.path, vid_rows.video)]))
print("frames read:", sum(f is not None for f in frames), "of", len(frames), "; frame size (pixels, every 3rd):", frames[0].shape)
fig, axs = plt.subplots(3, 14, figsize=(14, 8))
for ax in axs.flat:
    ax.axis("off")
for ax, f, (_, r) in zip(axs.flat, frames, vid_rows.iterrows()):
    ax.imshow(f, cmap="gray"); ax.set_title(f"{r.subject}\n{r.condition}", fontsize=7)
plt.tight_layout(); plt.show()

Checks of the description against the data:

- Sampling at 10 kHz in every sweep used, as in the Methods.
- In current clamp ch-1 is labelled volts, but its values are tens of pA, positive in most sweeps
  whose description names a positive holding current, and follow the 1 s steps from -120 to 260 pA:
  ch-1 is the injected current. The seal test is stored as an `IZeroClampSeries` but its ch-1 is a
  square voltage command of about 5 mV and its ch-0 a current: it is voltage clamp.
- The descriptions and the measured current disagree in places: 77 rest sweeps in 13 cells carry
  more than +20 pA (up to +108 pA; B16, described as -100 pA, carries +106 pA), and 11 firing sweeps in 6 cells
  carry less than +5 pA (C29, described as +35 pA, carries -57 pA). 148 of the 2437 ChR2 rest
  sweeps, in 28 cells, fire before the light. The Methods say cells at rest were held below
  threshold, so those sweeps are left out of the rest averages (C51 and C52 lose all theirs).
- 51 sweeps of one cell (P10) have a ch-1 far outside any current (beyond +-10 nA) and a membrane
  potential at 0 mV: they are failed sweeps and are left out. All 51 are rest sweeps; its other 28 rest
  sweeps (none firing before the light), its steps and its 13 firing sweeps are normal and are kept.
  In the first round the whole cell's rest average, with the failed sweeps in it, gave a baseline of -18 mV.
- The files give the sex of 3 subjects (23, C137 and C150; 17 ChR2 cells) as female, where the
  Methods say every mouse was male. Without these cells the dataset has 25 BLP and 26 CeA cells
  with current steps, fewer than the paper's 27 BLP cells in the clustering; with them 31 and 30.
  So the cells are kept, and male mice only is shown as a variant at the end. Either the sex in the
  files or the paper's statement is wrong for these cells, or these cells are not among the paper's:
  the mice labelled male give 48 BLP cells with rest sweeps, the paper's count (fig6df).
- Subject ids that differ only by a suffix ('4484' and '4484?', '235' and '235 cage', 'in_vivo_5'
  and 'in_vivo_5_2') are probably the same mice, which would make 17 subjects, not 20.
- The light time is not in the files. The assumed onset is where the responses start, and the
  following maxima of the grand average fall about 33 ms apart, as 30 Hz pulses would; the response
  at the assumed onset is far above the same measure 6 s later. The ChrimsonR sweeps, with red,
  blue or both, start at the same time (grand averages cross 10% at 5.473 to 5.476 s). This is
  consistent with the description of 8 pulses at 30 Hz but does not prove the onset to the millisecond.
- No text field of the files (sweep descriptions, session description, the stimulus types of the
  sequential recordings table, electrode descriptions) names TTX or 4-AP, so the TTX/4AP recordings
  of Figure 6 supplement 1D-E cannot be identified.
- Imaging: the traces are stated at 25 Hz, but 19,211 frames at 25 Hz last 768 s while the
  behaviour videos last 961 s (median; one lasts 800 s); at the Methods' 20 frames/s the imaging
  lasts 961 s, within 4 s of the video in every session but that one. The data support 20 Hz, and
  the stated rate is wrong. The virus, stimulus and experiment description fields of every imaging
  file describe another experiment (a GRIN lens in the mPFC, Chrimson in the BLA, 589 nm light at
  20 Hz), where the Methods give a lens over the CeA, Chrimson in the DRN and 620 nm light at 30 Hz.
- The number of ROIs is about twice the paper's cell counts in every condition (above), and the
  files do not mark which ROIs the paper kept.
- The behaviour videos show the arena from above with the mouse and both cups in every session; the
  cups are covered, so which one holds the juvenile cannot be seen in these frames.

## fig6df-epsp-ipsp-at-rest

> At the resting membrane potential, photostimulation of DRN-DAT terminals evoked both excitatory
> and inhibitory post-synaptic potentials in downstream cells of the BNST, CeA and BLP (19, 36 and
> 48 cells recorded).

Per ChR2 cell, the average of its rest sweeps (30 s light sweeps with no or negative holding
current and no action potential in the 5 s before the light; Methods). Peak and trough within
0-5.5 s of the first pulse against the 5 s baseline, detected beyond 2 or 3 SD of that baseline
(the criterion's readings); 5 SD is shown as a further variant. The same detection is run in a
light-free window of the same average, 12 s after the light, to show how often it finds a
deflection where there is no light. Cells recorded only with other protocols (20 s or 60 s
sweeps, or light at 10 s) are left out because their light time cannot be fixed; their number is
printed.

In [ ]:
REGIONS = ["BNST", "CeA", "BLP"]
EXPECTED_REST = {"BNST": 19, "CeA": 36, "BLP": 48}
KS = (2, 3, 5)          # the criterion's readings, 2 and 3 SD, and 5 SD as a variant
rest, all_spiking = [], []
for c in CELLS:
    if "ChR2" not in c["virus"] or "rest" not in c["sweeps"]:
        continue
    sub = de.subthreshold(de.valid(c["sweeps"]["rest"]))
    if not sub:
        all_spiking.append(c["cell_id"])
        continue
    avg = de.average(sub)
    row = {"path": c["path"], "subject": c["subject"], "sex": c["sex"], "cell": c["cell_id"], "region": c["region"],
           "n_sweeps": len(sub), "n_spiking_left_out": len(c["sweeps"]["rest"]) - len(sub)}
    for k in KS:
        e = de.evoked(avg, k)
        ctl = de.evoked(avg, k, onset=de.CONTROL_ONSET_S)       # light-free window, 12 s later
        row[f"class_k{k}"], row[f"epsp_k{k}"], row[f"ipsp_k{k}"] = de.response_class(e), e["epsp"], e["ipsp"]
        row[f"ctl_class_k{k}"], row[f"ctl_epsp_k{k}"], row[f"ctl_ipsp_k{k}"] = de.response_class(ctl), ctl["epsp"], ctl["ipsp"]
    row.update({kk: e[kk] for kk in ("baseline_mV", "sd_mV", "peak_mV", "trough_mV", "area_mVs")})
    row["peak_over_sd"], row["trough_over_sd"] = e["peak_mV"] / e["sd_mV"], -e["trough_mV"] / e["sd_mV"]
    row["ctl_peak_over_sd"], row["ctl_trough_over_sd"] = ctl["peak_mV"] / ctl["sd_mV"], -ctl["trough_mV"] / ctl["sd_mV"]
    row["evoked_ap"] = de.light_window(avg)[1].max() > de.SPIKE_THRESHOLD_MV
    rest.append(row)
REST = pd.DataFrame(rest)
no_rest = cells[(cells.opsin == "ChR2") & (cells.n_rest == 0)]
print("ChR2 cells without 30 s rest sweeps (left out):", len(no_rest), dict(no_rest.region.value_counts()))
print("ChR2 cells whose every rest sweep has spikes before the light (left out):", all_spiking)
print("cells of mice labelled female (kept):", dict(REST.loc[REST.sex != "M", "region"].value_counts()))
print("cells per region:", {r: int((REST.region == r).sum()) for r in REGIONS},
      "; male mice only:", {r: int((REST[REST.sex == "M"].region == r).sum()) for r in REGIONS}, "; expected", EXPECTED_REST)

def fig6df(R, k, window=""):
    tab = pd.crosstab(R.region, R[f"{window}class_k{k}"]).reindex(REGIONS).fillna(0).astype(int)
    ok = all(R[R.region == r][f"{window}epsp_k{k}"].any() for r in REGIONS) and R[f"{window}ipsp_k{k}"].any()
    return tab, ok

for k in KS:
    tab, ok = fig6df(REST, k)
    ctab, cok = fig6df(REST, k, "ctl_")
    print(f"\nreading k={k} SD{' (variant)' if k == 5 else ''}:"); print(tab)
    print("  percent:"); print((100 * tab.div(tab.sum(axis=1), axis=0)).round(1))
    print(f"  every region has a cell with an EPSP and some cell has an IPSP: {ok}")
    print(f"  the same detection in the light-free window 12 s later (no light):"); print(ctab)
    print(f"  ... would pass there too: {cok}")

# how often the rule detects a deflection with and without light, as the threshold rises
fp = []
for k in (2, 3, 4, 5, 6, 8, 10):
    fp.append({"k (SD)": k, "EPSP, light %": 100 * (REST.peak_over_sd > k).mean(), "EPSP, no light %": 100 * (REST.ctl_peak_over_sd > k).mean(),
               "IPSP, light %": 100 * (REST.trough_over_sd > k).mean(), "IPSP, no light %": 100 * (REST.ctl_trough_over_sd > k).mean()})
print(f"\npercent of the {len(REST)} cells with a detection, in the light window and in the light-free window:")
print(pd.DataFrame(fp).round(1).to_string(index=False))
print("\nby subject (k=3):")
print(pd.crosstab([REST.region, REST.subject], REST.class_k3))
print("\ncells whose rest average crosses 0 mV after the light (an evoked action potential), by region:",
      REST[REST.evoked_ap].groupby("region").cell.apply(list).to_dict())
print("\npeak (mV):", summary(REST.peak_mV), "\ntrough (mV):", summary(REST.trough_mV), "\nbaseline SD (mV):", summary(REST.sd_mV))

**Result: reproduced.** Under both readings every region has cells with an EPSP and cells with an IPSP: at 2 SD
the BNST, CeA and BLP have 15, 37 and 54 cells with an EPSP and 11, 30 and 49 with an IPSP; at 3 SD 13, 35 and 53,
and 6, 14 and 38. The test alone does not show this, because the same rule passes in the light-free window too.
The evidence is the contrast: at 3 SD 92.7% of cells have an EPSP and 53.2% an IPSP after the light, against 79.8%
and 30.3% without light; at 5 SD 84.4% and 19.3% against 40.4% and 4.6%. The dataset holds 15, 38 and 56 ChR2 cells
with usable rest sweeps where the paper reports 19, 36 and 48 (14, 35 and 48 from the mice labelled male), so the
counts do not match; 19 ChR2 cells recorded only with other light protocols are left out. At 2 SD most cells are
mix (73 to 84%); at 5 SD most are excitation only (62 to 74%). For IPSPs the contrast is weak at 3 SD (53.2%
against 30.3%) and clear from 4 SD up (27.5% against 11.0%).

## fig6gi-firing-responses

> In cells made to fire by constant current, photostimulation increased firing in all BNST cells
> (5 of 5), while CeA (20 cells) and BLP (17 cells) cells showed either an increase or a decrease in
> firing.

Firing sweeps (set a) are the 30 s light sweeps with a positive holding current in their
description (Methods). Spikes are upward crossings of 0 mV. IEI before (5 s, or 5.5 s as the
Figure 6 supplement 1G variant) and 5 s after the first pulse; readings (a) mean of all IEIs pooled
over sweeps, (b) mean over sweeps of each sweep's mean IEI. As a variant, set (b) adds the rest
sweeps that fire before the light with a measured current above +10 pA, since the descriptions and
the measured current disagree (data check).

In [ ]:
EXPECTED_FIRE = {"BNST": 5, "CeA": 20, "BLP": 17}

def fire_row(c, sweeps, set_name):
    row = {"path": c["path"], "subject": c["subject"], "sex": c["sex"], "cell": c["cell_id"], "region": c["region"],
           "set": set_name, "n_sweeps": len(sweeps), "max_vm_mV": max(float(s["v"].max()) for s in sweeps),
           "spikes": sum(len(de.spike_times(s["v"])) for s in sweeps)}
    for before in (5.0, 5.5):
        r = de.iei_change(sweeps, before_s=before)
        for rd in ("a", "b"):
            row[f"{rd}_{before}_before"], row[f"{rd}_{before}_after"] = r[f"{rd}_before"], r[f"{rd}_after"]
            row[f"label_{rd}_{before}"] = de.iei_label(r[f"{rd}_before"], r[f"{rd}_after"])
    return row

fire, fire_b = [], []
for c in CELLS:
    if "ChR2" not in c["virus"]:
        continue
    if "firing" in c["sweeps"]:
        fire.append(fire_row(c, de.valid(c["sweeps"]["firing"]), "a"))
    # variant set (b): also the 'rest' sweeps that fire before the light with a measured current above +10 pA
    extra = [s for s in de.valid(c["sweeps"].get("rest", [])) if s["hold_pA"] > 10
             and len(de.spike_times(s["v"][i0 - 50_000:i0])) > 0]
    if "firing" in c["sweeps"] or extra:
        fire_b.append(fire_row(c, de.valid(c["sweeps"].get("firing", [])) + extra, "b"))
FIRE = pd.DataFrame(fire)
FIRE_B = pd.DataFrame(fire_b)

def fig6gi(F, name):
    rows = []
    for before in (5.0, 5.5):
        for rd in ("a", "b"):
            col = f"label_{rd}_{before}"
            lab_bnst = F[F.region == "BNST"][col]
            rows.append({"set": name, "reading": rd, "baseline_s": before,
                         **{f"{r}_exc/inh/none": f"{(F[F.region == r][col] == 'excitation').sum()}/{(F[F.region == r][col] == 'inhibition').sum()}/{F[F.region == r][col].isna().sum()}" for r in REGIONS},
                         "every BNST cell excited": len(lab_bnst) > 0 and (lab_bnst == "excitation").all(),
                         "every BNST cell with an IEI excited": lab_bnst.notna().any() and (lab_bnst.dropna() == "excitation").all(),
                         "both classes in CeA and BLP": all(set(F[F.region == r][col]) >= {"excitation", "inhibition"} for r in ("CeA", "BLP"))})
    return pd.DataFrame(rows)

print("cells per region, firing sweeps by description (a):", {r: int((FIRE.region == r).sum()) for r in REGIONS},
      "; male mice only:", {r: int((FIRE[FIRE.sex == "M"].region == r).sum()) for r in REGIONS},
      "; set (b):", {r: int((FIRE_B.region == r).sum()) for r in REGIONS}, "; expected", EXPECTED_FIRE)
print("cells of mice labelled female (kept):", FIRE[FIRE.sex != "M"].cell.tolist())
GI = pd.concat([fig6gi(FIRE, "a"), fig6gi(FIRE_B, "b (variant)")])
print(GI.to_string(index=False))
print("\ncells with no IEI label (reading a, 5 s), with their spikes and the highest Vm of their firing sweeps:")
print(FIRE[FIRE["label_a_5.0"].isna()][["region", "cell", "n_sweeps", "spikes", "max_vm_mV"]].round(1).to_string(index=False))
print("cells added in set (b):", sorted(set(FIRE_B.cell) - set(FIRE.cell)))
FIRE["ratio_a5"] = FIRE["a_5.0_after"] / FIRE["a_5.0_before"]
print("\nper cell, IEI after/before, reading (a), 5 s:")
print(FIRE[["region", "subject", "cell", "n_sweeps", "a_5.0_before", "a_5.0_after", "ratio_a5", "label_a_5.0"]].round(3).sort_values("region").to_string())

**Result: partially reproduced.** In every reading both excitation and inhibition occur in the CeA (8 and 2) and
the BLP (4 to 6 and 7 to 9), and the BNST cells that can be classed are excited (2 of 2). The third BNST cell (B7, one
sweep) has no action potential (highest Vm -29 mV), so the paper's 5 of 5 cannot be checked: the dataset holds 3,
13 and 13 cells with firing sweeps where the paper reports 5, 20 and 17. Three CeA cells (C29, C39 and C47) and one
BLP cell (P9) also lack an IEI before or after the light; C29 and C47 never cross 0 mV (highest Vm -59 and -49 mV),
so a lower spike threshold would not change this. In set (b), 5, 17 and 17 cells, the BNST cells with an IEI are
excited in three of the four readings.

## fig6j-epsp-amplitude-region

> The peak amplitude of the optically evoked EPSP differed between regions, larger in the CeA than in
> the BLP.

EPSP peak of the average rest trace, one value per cell with an EPSP; cells (a) excitation and mix,
(b) excitation only; under each detection reading of fig6df (2 and 3 SD), with 5 SD as a variant.
Kruskal-Wallis, then Dunn's test (Prism's adjustment).

In [ ]:
def fig6j(k, which, R=None):
    R = REST if R is None else R
    cls = R[f"class_k{k}"]
    sel = R[cls.isin(["excitation", "mix"] if which == "a" else ["excitation"])]
    g = {r: sel[sel.region == r].peak_mV.values for r in REGIONS}
    out = {"k": k, "cells": which, **{f"n_{r}": len(g[r]) for r in REGIONS}}
    if min(len(v) for v in g.values()) < 2:
        return {**out, "H": np.nan, "p": np.nan}
    H, p = stats.kruskal(*g.values())
    dn = de.dunn(g)
    return {**out, **{f"median_{r}": np.median(g[r]) for r in REGIONS}, "H": H, "p": p,
            "dunn_CeA_BLP": dn[("CeA", "BLP")], "dunn_BNST_CeA": dn[("BNST", "CeA")], "dunn_BNST_BLP": dn[("BNST", "BLP")]}

def j_table(R=None):
    J = pd.DataFrame([fig6j(k, w, R) for k in KS for w in ("a", "b")])
    J["holds"] = (J.p < 0.05) & (J.dunn_CeA_BLP < 0.05) & (J.median_CeA > J.median_BLP)
    return J
J = j_table()
print(J.round(4).to_string())
print("rows with k=5 are a variant; the criterion's readings are k=2 and 3")
print("\nvariant: reading (a) without the cells whose average crosses 0 mV after the light (an evoked action potential)")
print(j_table(REST[~REST.evoked_ap]).query("cells == 'a'").round(4).to_string())
print("\npaper: H = 6.790, p = 0.0335; Dunn CeA vs BLP p = 0.0378")

**Result: partially reproduced.** The CeA median exceeds the BLP median in five of the six rows, but the test is
significant in only one of the criterion's four readings: reading (a) at 3 SD, narrowly (H = 6.02, p = 0.049; Dunn
CeA against BLP p = 0.043). Reading (a) at 2 SD gives p = 0.088, and reading (b) p = 0.86 and 0.27. At 2 and 3 SD
most excitation cells are classed mix because the rule finds an IPSP in 67% and 30% of the light-free windows, so
reading (b) keeps only 19 and 48 cells. The one passing reading rests on one cell: without C15, whose
rest average carries an evoked action potential, reading (a) at 3 SD gives p = 0.075 (Dunn p = 0.071). At 5 SD, where that rate is 4.6%, both readings are significant (p = 0.016
and 0.037), and reading (a) stays so without C15 (p = 0.026). With the cells of the mice labelled male only, reading (a) is significant at 2 and 3 SD (variant at the
end).

## fig6k-ipsp-amplitude-region

> The trough amplitude of the optically evoked IPSP differed between regions, deeper in the CeA than
> in the BLP.

IPSP trough of the average rest trace; cells (a) inhibition and mix, (b) inhibition only; detection
at 2 or 3 SD, with 5 SD as a variant. One-way ANOVA, then Tukey's test and Holm-Sidak (the post
hoc test is not named).

In [ ]:
def fig6k(k, which, R=None):
    R = REST if R is None else R
    cls = R[f"class_k{k}"]
    sel = R[cls.isin(["inhibition", "mix"] if which == "a" else ["inhibition"])]
    g = {r: sel[sel.region == r].trough_mV.values for r in REGIONS if (sel.region == r).sum() > 0}
    out = {"k": k, "cells": which, "n_total": len(sel), **{f"n_{r}": int((sel.region == r).sum()) for r in REGIONS},
           **{f"mean_{r}": sel[sel.region == r].trough_mV.mean() for r in REGIONS}}
    usable = {r: v for r, v in g.items() if len(v) >= 2}
    if len(usable) < 2 or "CeA" not in usable or "BLP" not in usable:
        return {**out, "F": np.nan, "p": np.nan}
    F, p = stats.f_oneway(*usable.values())
    tk = pairwise_tukeyhsd(np.concatenate(list(usable.values())), np.concatenate([[r] * len(v) for r, v in usable.items()]))
    res = pd.DataFrame(tk._results_table.data[1:], columns=tk._results_table.data[0])
    t_cea_blp = float(res[((res.group1 == "BLP") & (res.group2 == "CeA")) | ((res.group1 == "CeA") & (res.group2 == "BLP"))]["p-adj"].iloc[0])
    hs = de.holm_sidak(usable)
    hs_cea_blp = hs.get(("CeA", "BLP"), hs.get(("BLP", "CeA")))
    return {**out, "groups_in_test": len(usable), "df": f"{len(usable) - 1},{sum(len(v) for v in usable.values()) - len(usable)}",
            "F": F, "p": p, "tukey_CeA_BLP": t_cea_blp, "holm_sidak_CeA_BLP": hs_cea_blp}

def k_table(R=None):
    K = pd.DataFrame([fig6k(k, w, R) for k in KS for w in ("a", "b")])
    K["holds_tukey"] = (K.p < 0.05) & (K.tukey_CeA_BLP < 0.05) & (K.mean_CeA < K.mean_BLP)
    K["holds_holm_sidak"] = (K.p < 0.05) & (K.holm_sidak_CeA_BLP < 0.05) & (K.mean_CeA < K.mean_BLP)
    K["n_within_34_pm_3.4"] = (K.n_total - 34).abs() <= 3.4
    return K
K = k_table()
print(K.round(4).to_string())
print("rows with k=5 are a variant; the criterion's readings are k=2 and 3")
print("\npaper: F(2,31) = 8.150, p = 0.0014, CeA vs BLP p = 0.0014; 34 cells")

**Result: partially reproduced, under the 3 SD reading.** The readings disagree. At 2 SD the ANOVA on 90 cells gives
p = 0.76. At 3 SD the ANOVA on 58 cells gives F(2,55) = 6.88, p = 0.0021, and CeA against BLP Tukey p = 0.0031 and
Holm-Sidak p = 0.0033, with the CeA mean trough deeper (-0.78 against -0.45 mV). At 5 SD (21 cells, one BNST cell, so
two groups) the CeA trough is also deeper (p < 0.0001). The 3 SD reading is adopted because it agrees with 5 SD, the
threshold at which IPSPs separate from noise (19.3% of cells after the light against 4.6% without, fig6df). The
false-detection rate alone does not settle it: at 3 SD the rule finds an IPSP in 30.3% of light-free windows against
53.2% after the light, so more than half of the 3 SD IPSPs may be noise, as most of the 2 SD ones are (67.0% against
82.6%). The paper's test has 34 cells; 58 is outside 34 +- 3.4. Reading (b) leaves 3 to 5 cells and cannot be tested.


## fig6-s1de-monosynaptic (blocked)

> The optically evoked EPSP and the slow component of the IPSP persisted after TTX/4AP, so the
> responses were typically monosynaptic.

No text field of the files names TTX or 4-AP (data check: the 103 sweep descriptions, the session
descriptions, the 103 stimulus types of the sequential recordings table and the electrode
descriptions; `general/` has no notes, protocol, slices or pharmacology field). The sweeps recorded
after TTX/4AP cannot be told from those before. Blocked, `metadata_insufficient`.

## Baseline properties for the clustering claims

Per ChR2 cell from its 1 s current steps (Methods): voltage sag at -120 pA (steady state the last
100 ms, or 200 ms, of the step), ramp ratio at the largest subthreshold step (absolute Vm as
written, or deflections from the pre-step Vm), firing delay at rheobase, and the maximum
instantaneous frequency in the first 100 ms of the depolarizing steps (0 Hz when no two spikes fall
in that time). Capacitance by the Q-method from the 5 mV seal-test pulses, the only voltage-clamp
test pulses in the files (the Methods use a 5 mV, 250 ms step); no correction for the series
and membrane resistance is made, so the error of this estimate depends on each cell's own resistances. Some BLP cells
were also stepped from 220 pA up to 600 pA; those steps are used for rheobase.

Response label per cell (Methods): from the IEI when the cell has firing sweeps, otherwise
excitation for an EPSP only, inhibition for an IPSP only, the sign of the 0-5.5 s voltage area for
a mix, and none otherwise. Labels are made at 2 and 3 SD (the criterion's readings) and at 5 SD
(a variant).

In [ ]:
feat_rows = []
for c in CELLS:
    if "ChR2" not in c["virus"]:
        continue
    row = {"path": c["path"], "subject": c["subject"], "sex": c["sex"], "cell": c["cell_id"], "region": c["region"]}
    if "steps" in c["sweeps"]:
        for ss in (100, 200):
            f = de.step_features(c["sweeps"]["steps"], sag_ss_ms=ss) or {}
            row[f"sag_{ss}"] = f.get("sag_mV", np.nan)
        row.update({"ramp_abs": f.get("ramp_ratio", np.nan), "ramp_defl": f.get("ramp_ratio_deflection", np.nan),
                    "delay": f.get("firing_delay_ms", np.nan), "maxfreq": f.get("max_freq_hz", np.nan),
                    "step_min": f.get("min_pA", np.nan), "step_max": f.get("max_pA", np.nan)})
    if "seal" in c["sweeps"]:
        row["cap"] = de.cell_capacitance_pf(c["sweeps"]["seal"])
    feat_rows.append(row)
FEAT = pd.DataFrame(feat_rows)
FEAT = FEAT.merge(REST[["path", "peak_mV", "trough_mV", "area_mVs", *[f"{x}_k{k}" for k in KS for x in ("class", "epsp")]]],
                          on="path", how="left")
FEAT = FEAT.merge(FIRE[["path", "label_a_5.0", "label_b_5.0"]], on="path", how="left")

def label(row, k, iei):
    lab = row.get(f"label_{iei}_5.0")
    if isinstance(lab, str):
        return lab
    cls = row.get(f"class_k{k}")
    if cls == "mix":
        return "excitation" if row["area_mVs"] > 0 else "inhibition"
    return {"excitation": "excitation", "inhibition": "inhibition", "none": "none"}.get(cls, "no data")
for k in KS:
    for iei in ("a", "b"):
        FEAT[f"resp_k{k}_{iei}"] = FEAT.apply(lambda r: label(r, k, iei), axis=1)
        FEAT[f"src_k{k}_{iei}"] = np.where(FEAT[f"label_{iei}_5.0"].notna(), "IEI",
                                               np.where(FEAT[f"class_k{k}"].notna(), "rest " + FEAT[f"class_k{k}"].astype(str), "no data"))
print("ChR2 cells with current steps:", int(FEAT.step_min.notna().sum()), "; with each feature, by region:")
print(FEAT.groupby("region")[["sag_100", "ramp_abs", "delay", "maxfreq", "cap"]].count())
for col in ["sag_100", "sag_200", "ramp_abs", "ramp_defl", "delay", "maxfreq", "cap"]:
    print(f"{col}: {summary(FEAT[col])}")

In [ ]:
CEA_FEATS = ["ramp", "maxfreq", "delay", "sag"]
BLP_FEATS = ["cap", "maxfreq", "delay", "sag"]
CRIT_K = (2, 3)         # the criterion's detection readings; k=5 is a variant

def cluster(df, cols):
    X = df[cols].to_numpy(float)
    X = (X - X.min(0)) / (X.max(0) - X.min(0))
    return fcluster(linkage(X, method="ward", metric="euclidean"), 2, criterion="maxclust")

def frame_for(region, ramp, sag, feats, F=None):
    F = FEAT if F is None else F
    df = F[F.region.isin([region] if isinstance(region, str) else region)].copy()
    df["ramp"], df["sag"] = df[ramp], df[sag]
    return df.dropna(subset=feats).reset_index(drop=True)

## fig6no-cea-clusters

> Hierarchical clustering of CeA cells on four baseline properties gave two clusters, late-firing
> cells (cluster 1) that were excited by DRN-DAT input and regular-firing cells (cluster 2) that were
> mostly inhibited.

Max-min normalized ramp ratio, maximum instantaneous frequency, firing delay and sag; Ward linkage
on Euclidean distance; two clusters. The cluster with the longer mean firing delay is the late-firing
cluster. Every reading of the criterion is run: ramp ratio absolute or from deflections, sag
steady state 100 or 200 ms, and the detection (2 or 3 SD) and IEI readings behind the labels; 5 SD
labels are a variant.

In [ ]:
def cea_runs(F=None):
    runs, main = [], None
    for ramp in ("ramp_abs", "ramp_defl"):
        for sag in ("sag_100", "sag_200"):
            df = frame_for("CeA", ramp, sag, CEA_FEATS, F)
            df["cl"] = cluster(df, CEA_FEATS)
            df["late"] = df.cl == df.groupby("cl").delay.mean().idxmax()
            for k in KS:
                for iei in ("a", "b"):
                    lab = df[f"resp_k{k}_{iei}"]
                    L, R = lab[df.late], lab[~df.late]
                    runs.append({"ramp": ramp, "sag": sag, "k": k, "iei": iei, "n": len(df),
                                 "n_late": len(L), "late_excited": (L == "excitation").sum(), "late_inhibited": (L == "inhibition").sum(),
                                 "late_none_or_nodata": L.isin(["none", "no data"]).sum(),
                                 "n_regular": len(R), "regular_excited": (R == "excitation").sum(),
                                 "regular_inhibited": (R == "inhibition").sum(),
                                 "regular_none_or_nodata": R.isin(["none", "no data"]).sum(),
                                 "late_all_excited": (L == "excitation").all(),
                                 "regular_over_half_inhibited": (R == "inhibition").sum() > len(R) / 2,
                                 "holds": (L == "excitation").all() and (R == "inhibition").sum() > len(R) / 2})
            if ramp == "ramp_abs" and sag == "sag_100":
                main = df.copy()
    return pd.DataFrame(runs), main
CEA_RUNS, CEA_MAIN = cea_runs()
print(CEA_RUNS.to_string())
print("rows with k=5 are a variant; the criterion's readings are k=2 and 3")
print("\nthe criterion's two parts, number of readings in which each holds (of 8 per k):")
print(CEA_RUNS.groupby("k")[["late_all_excited", "regular_over_half_inhibited", "holds"]].sum().to_string())
print("\ncells clustered:", CEA_RUNS.n.unique(), "(expected 26); subjects:", CEA_MAIN.subject.nunique())
print("\nfirst reading, cluster means of the raw features:")
print(CEA_MAIN.groupby("late")[CEA_FEATS].mean().round(3))
print("\nper cell of the late-firing cluster (first reading): label at each k (IEI reading a), its source, area and peak")
cols = ["subject", "cell"] + [f"resp_k{k}_a" for k in KS] + [f"src_k{k}_a" for k in KS] + ["area_mVs", "peak_mV", "trough_mV"]
print(CEA_MAIN[CEA_MAIN.late][cols].round(3).sort_values("resp_k3_a").to_string(index=False))
print("\nlabels by cluster and subject, first reading (k=3, IEI a):")
print(pd.crosstab([CEA_MAIN.late, CEA_MAIN.subject], CEA_MAIN.resp_k3_a))

**Result: partially reproduced (verified outcome); not reproduced is proposed.** Every reading gives two clusters of
27 cells (the paper has 26), a late-firing cluster (mean firing delay 415 against 168 ms) and a regular-firing one.
The late-firing cluster is mostly excited (11 to 13 of 17 or 18 cells) but holds 2 or 3 inhibited cells (C20, a mix
with a negative area, and C53, from its IEI; C41 too at 2 SD) and 3 cells without response data, so not every cell
is excited as the criterion requires; 6 of the 9 or 10 regular-firing cells are inhibited, more than half. The
readings agree: the first part of the criterion fails in all 16 readings at 2 and 3 SD and the second holds in all
16. With 5 SD labels the late-firing cluster has one inhibited cell (C53) and only 3 of the regular-firing cells are
inhibited, so both parts fail. The criterion has no partial clause (the claims file gives one where it means one,
as for fig6pq), and its first part fails in every reading, so not reproduced fits the criterion better than the
verified partially reproduced. The change is proposed for the next verification round.


## fig6-s1h-cea-cluster-properties

> The two CeA clusters differed in all four input properties (ramp ratio, maximum instantaneous
> frequency, firing delay and voltage sag).

Unpaired two-sided t-tests between the clusters of fig6no-cea-clusters on the raw features, under
each clustering reading.

In [ ]:
def h_table(F=None):
    rows = []
    for ramp in ("ramp_abs", "ramp_defl"):
        for sag in ("sag_100", "sag_200"):
            df = frame_for("CeA", ramp, sag, CEA_FEATS, F)
            df["cl"] = cluster(df, CEA_FEATS)
            a, b = df[df.cl == 1], df[df.cl == 2]
            row = {"ramp": ramp, "sag": sag, "n1": len(a), "n2": len(b)}
            for f in CEA_FEATS:
                t, p = stats.ttest_ind(a[f], b[f])
                row[f"t_{f}"], row[f"p_{f}"] = t, p
            row["all_p<0.05"] = all(row[f"p_{f}"] < 0.05 for f in CEA_FEATS)
            rows.append(row)
    return pd.DataFrame(rows)
H = h_table()
print(H.round(5).to_string())
print("\npaper: t24 = 3.502 (ramp), 4.698 (max freq), 5.050 (delay), 3.983 (sag)")

**Result: reproduced.** Under every reading the two CeA clusters differ in all four properties (ramp ratio p = 0.0072
to 0.029, maximum frequency and sag p below 0.0001, firing delay p = 0.0089 to 0.014), with |t(25)| from 2.32 to 6.32.

## fig6pq-blp-clusters

> BLP cells clustered into putative pyramidal neurons (cluster 1) and putative GABAergic interneurons
> (cluster 2); 93% of the putative pyramidal neurons were inhibited by DRN-DAT input and 62% of the
> putative interneurons were excited.

As for the CeA with capacitance in place of the ramp ratio. The pyramidal cluster is the one with
the larger mean capacitance; whether it also has the lower maximum frequency is printed.
Percentages over all cells of a cluster.

In [ ]:
def blp_runs(F=None):
    runs, main = [], None
    for sag in ("sag_100", "sag_200"):
        df = frame_for("BLP", "ramp_abs", sag, BLP_FEATS, F)
        df["cl"] = cluster(df, BLP_FEATS)
        means = df.groupby("cl")[["cap", "maxfreq"]].mean()
        pyr = means.cap.idxmax()
        df["pyr"] = df.cl == pyr
        for k in KS:
            for iei in ("a", "b"):
                lab = df[f"resp_k{k}_{iei}"]
                P, I = lab[df.pyr], lab[~df.pyr]
                pi, ie = 100 * (P == "inhibition").mean(), 100 * (I == "excitation").mean()
                runs.append({"sag": sag, "k": k, "iei": iei, "n": len(df), "n_pyr": len(P), "n_int": len(I),
                             "pyr_has_lower_maxfreq": means.maxfreq[pyr] < means.maxfreq.drop(pyr).iloc[0],
                             "pyr_inhibited_%": pi, "pyr_excited_%": 100 * (P == "excitation").mean(),
                             "int_excited_%": ie, "int_inhibited_%": 100 * (I == "inhibition").mean(),
                             "within_tol": abs(pi - 93) <= 9.3 and abs(ie - 62) <= 6.2,
                             "direction": (P == "inhibition").mean() > 0.5 and (I == "excitation").mean() > 0.5})
        if sag == "sag_100":
            main = df.copy()
    return pd.DataFrame(runs), main
BLP_RUNS, BLP_MAIN = blp_runs()
print(BLP_RUNS.round(1).to_string())
print("rows with k=5 are a variant; the criterion's readings are k=2 and 3")
print("\ncells clustered:", BLP_RUNS.n.unique(), "(expected 27)")
print("\nfirst reading, cluster means of the raw features:")
print(BLP_MAIN.groupby("pyr")[BLP_FEATS].mean().round(2))
print("\nlabels by cluster (k=3, IEI a):")
print(pd.crosstab(BLP_MAIN.pyr, BLP_MAIN.resp_k3_a))
print("\nby subject:")
print(pd.crosstab([BLP_MAIN.pyr, BLP_MAIN.subject], BLP_MAIN.resp_k3_a))

**Result: partially reproduced.** Every reading gives 28 BLP cells (the paper has 27) in a cluster of larger
capacitance and lower maximum frequency (17 cells, the putative pyramidal neurons; 77 against 37 pF, 27 against
133 Hz) and one of 11. 64.7% (2 SD) or 58.8% (3 SD) of the pyramidal cluster are inhibited (expected 93 ± 9.3%)
and 72.7% of the other cluster are excited (expected 62 ± 6.2%): the direction holds, the percentages are outside
tolerance. With 5 SD labels the direction fails: 41.2% of the pyramidal cluster are inhibited and 52.9% excited.

## fig6-s1i-blp-cluster-properties

> The two BLP clusters differed in capacitance, maximum instantaneous frequency, firing delay and
> voltage sag.

In [ ]:
def i_table(F=None):
    rows = []
    for sag in ("sag_100", "sag_200"):
        df = frame_for("BLP", "ramp_abs", sag, BLP_FEATS, F)
        df["cl"] = cluster(df, BLP_FEATS)
        a, b = df[df.cl == 1], df[df.cl == 2]
        row = {"sag": sag, "n1": len(a), "n2": len(b)}
        for f in BLP_FEATS:
            t, p = stats.ttest_ind(a[f], b[f])
            row[f"t_{f}"], row[f"p_{f}"] = t, p
        row["all_p<0.05"] = all(row[f"p_{f}"] < 0.05 for f in BLP_FEATS)
        rows.append(row)
    return pd.DataFrame(rows)
I_ = i_table()
print(I_.round(5).to_string())
# how much the split depends on single cells: each BLP cell left out in turn (first sag reading)
blp_paths = frame_for("BLP", "ramp_abs", "sag_100", BLP_FEATS).path
loo = []
for pth in blp_paths:
    r = i_table(FEAT[FEAT.path != pth]).iloc[0]
    loo.append({"left_out": FEAT.loc[FEAT.path == pth, "cell"].iloc[0], "n1": r.n1, "n2": r.n2,
                **{f"p_{f}": r[f"p_{f}"] for f in BLP_FEATS}, "all_p<0.05": bool(r["all_p<0.05"])})
LOO = pd.DataFrame(loo)
print(f"\neach of the {len(LOO)} BLP cells left out in turn (sag_100): all four p < 0.05 in {LOO['all_p<0.05'].sum()} of {len(LOO)}")
print(LOO[~LOO["all_p<0.05"] | (LOO.left_out == "P10")].round(4).to_string(index=False))
print("\npaper: t25 = 4.803 (capacitance), 15.48 (max freq), 2.743 (delay), 2.705 (sag)")

**Result: reproduced.** Under both readings the two BLP clusters differ in capacitance (t(26) = 3.69, p = 0.0011),
maximum frequency (p below 0.0001), firing delay (p = 0.0005) and sag (p = 0.0021 and 0.0032). The split depends on
single cells: leaving out each of the 28 cells in turn, all four tests stay significant in 24 cases; without P10,
P19, P29 or P43 the clusters have 12 or 13 and 14 or 15 cells and the delay no longer differs (p = 0.053 to 0.063).


## fig6-s1jk-cluster-voltage-area

> The total voltage area evoked by DRN-DAT stimulation differed between the two clusters in both the
> CeA and the BLP, and the EPSP peak amplitude differed between the BLP clusters but not between the
> CeA clusters.

Total voltage area 0-5.5 s against the 5 s baseline of the average rest trace (Methods);
Mann-Whitney U for the area, unpaired t-test for the EPSP peak of cells with an EPSP, between the
clusters of each clustering reading.

In [ ]:
def jk(df, grp, region, reading):
    out = {"region": region, "reading": reading}
    a, b = df[df[grp]], df[~df[grp]]
    aa, bb = a.area_mVs.dropna(), b.area_mVs.dropna()
    out["n_area"] = f"{len(aa)}+{len(bb)}"
    out["U"], out["p_area"] = stats.mannwhitneyu(aa, bb, alternative="two-sided") if len(aa) and len(bb) else (np.nan, np.nan)
    out["median_area_1"], out["median_area_2"] = aa.median(), bb.median()
    for k in KS:
        ea = a[a[f"epsp_k{k}"] == True].peak_mV
        eb = b[b[f"epsp_k{k}"] == True].peak_mV
        t, p = stats.ttest_ind(ea, eb) if len(ea) > 1 and len(eb) > 1 else (np.nan, np.nan)
        out[f"n_epsp_k{k}"], out[f"t_epsp_k{k}"], out[f"p_epsp_k{k}"] = f"{len(ea)}+{len(eb)}", t, p
    return out

def jk_table(F=None):
    rows = []
    for ramp in ("ramp_abs", "ramp_defl"):
        for sag in ("sag_100", "sag_200"):
            df = frame_for("CeA", ramp, sag, CEA_FEATS, F)
            df["cl"] = cluster(df, CEA_FEATS)
            df["g"] = df.cl == df.groupby("cl").delay.mean().idxmax()
            rows.append(jk(df, "g", "CeA (late vs regular)", f"{ramp}, {sag}"))
    for sag in ("sag_100", "sag_200"):
        df = frame_for("BLP", "ramp_abs", sag, BLP_FEATS, F)
        df["cl"] = cluster(df, BLP_FEATS)
        df["g"] = df.cl == df.groupby("cl").cap.mean().idxmax()
        rows.append(jk(df, "g", "BLP (pyramidal vs interneuron)", sag))
    return pd.DataFrame(rows)

def jk_holds(JK, k):
    cea, blp = JK[JK.region.str.startswith("CeA")], JK[JK.region.str.startswith("BLP")]
    return bool((cea.p_area < 0.05).all() and (blp.p_area < 0.05).all() and (blp[f"p_epsp_k{k}"] < 0.05).all() and (cea[f"p_epsp_k{k}"] >= 0.05).all())
JK = jk_table()
print(JK.round(4).to_string())
for k in KS:
    print(f"k={k}{' (variant)' if k == 5 else ''}: holds in every reading:", jk_holds(JK, k))
print("\npaper: area CeA U = 22, p = 0.0023; BLP U = 29, p = 0.0019; EPSP CeA t17 = 1.40, p = 0.180; BLP t22 = 2.34, p = 0.029")

**Result: reproduced, with the cells of the mice labelled female.** Under every reading the total voltage area
differs between the CeA clusters (Mann-Whitney U = 97 or 101, p = 0.0001 or below) and between the BLP clusters
(U = 38, p = 0.020), the EPSP peak differs between the BLP clusters (t-test p = 0.023 at 2 and 3 SD) and not between
the CeA clusters (p = 0.097 to 0.21). With the mice labelled male only the BLP tests fail (area p = 0.11, EPSP peak
p = 0.099) while the CeA tests still hold (variant at the end), so the outcome rests on keeping the cells labelled
female.


## fig6-s1ln-combined-clusters

> Clustering CeA and BLP cells together gave two clusters with a response pattern very similar to
> that of the separate clusterings.

Five features (ramp ratio, capacitance, maximum frequency, firing delay, sag), max-min normalized,
Ward, two clusters, on the CeA and BLP cells with all five. The criterion asks that the excited
cluster be the same in both regions. Figure 6 supplement 1N, as drawn, shows the CeA cells of
cluster 1 mostly excited and its BLP cells inhibited, and the reverse in cluster 2; that pattern is
checked as a variant (`figure_s1n_pattern`), and whether the criterion misreads the figure is asked
of a reader (questions/001195.yaml).

In [ ]:
FIVE = ["ramp", "cap", "maxfreq", "delay", "sag"]
from scipy.cluster.hierarchy import dendrogram

def ln_table(F=None):
    rows, comb = [], None
    for ramp in ("ramp_abs", "ramp_defl"):
        for sag in ("sag_100", "sag_200"):
            df = frame_for(["CeA", "BLP"], ramp, sag, FIVE, F)
            df["cl"] = cluster(df, FIVE)
            for k in KS:
                for iei in ("a", "b"):
                    lab = df[f"resp_k{k}_{iei}"]
                    row = {"ramp": ramp, "sag": sag, "k": k, "iei": iei, "n": len(df)}
                    exc_cluster = {}
                    for reg in ("CeA", "BLP"):
                        for cl in (1, 2):
                            s = lab[(df.region == reg) & (df.cl == cl)]
                            row[f"{reg}_c{cl}"] = f"{(s == 'excitation').sum()}e/{(s == 'inhibition').sum()}i/{len(s)}"
                            row[f"{reg}_c{cl}_exc_frac"] = (s == "excitation").mean() if len(s) else np.nan
                            row[f"{reg}_c{cl}_inh_frac"] = (s == "inhibition").mean() if len(s) else np.nan
                        e = [cl for cl in (1, 2) if row[f"{reg}_c{cl}_exc_frac"] > 0.5]
                        i = [cl for cl in (1, 2) if row[f"{reg}_c{cl}_inh_frac"] > 0.5]
                        exc_cluster[reg] = e[0] if len(e) == 1 and len(i) == 1 and e != i else None
                    split = exc_cluster["CeA"] is not None and exc_cluster["BLP"] is not None
                    # the criterion: the excited cluster the same in both regions
                    row["holds"] = split and exc_cluster["CeA"] == exc_cluster["BLP"]
                    # Figure 6-S1N as drawn: CeA excited and BLP inhibited in one cluster, the reverse in the other
                    row["figure_s1n_pattern"] = split and exc_cluster["CeA"] != exc_cluster["BLP"]
                    rows.append(row)
            if ramp == "ramp_abs" and sag == "sag_100":
                comb = df.copy()
    return pd.DataFrame(rows), comb
LN, COMB = ln_table()
print(LN[[c for c in LN.columns if not c.endswith("frac")]].to_string())
print("rows with k=5 are a variant; the criterion's readings are k=2 and 3")
# agreement with the separate clusterings (first reading)
sep = pd.concat([CEA_MAIN[["path"]].assign(sep=np.where(CEA_MAIN.late, "CeA late", "CeA regular")),
                 BLP_MAIN[["path"]].assign(sep=np.where(BLP_MAIN.pyr, "BLP pyramidal", "BLP interneuron"))])
m = COMB.merge(sep, on="path")
print("\ncombined cluster against separate cluster (first reading):")
print(pd.crosstab(m.sep, m.cl))

# the combined dendrogram with region and response (k=3, IEI a) under each leaf, as Figure 6-S1M draws it
X = COMB[FIVE].to_numpy(float); X = (X - X.min(0)) / (X.max(0) - X.min(0))
fig, (a0, a1) = plt.subplots(2, 1, figsize=(12, 4), gridspec_kw={"height_ratios": [4, 1]}, sharex=True)
d = dendrogram(linkage(X, method="ward"), ax=a0, no_labels=True, color_threshold=0)
order = d["leaves"]
reg_c = {"CeA": "tab:pink", "BLP": "tab:blue"}
resp_c = {"excitation": "black", "inhibition": "0.6", "none": "white", "no data": "white"}
for j, idx in enumerate(order):
    x = 5 + 10 * j
    a1.add_patch(plt.Rectangle((x - 4, 1), 8, 0.8, color=reg_c[COMB.region[idx]]))
    a1.add_patch(plt.Rectangle((x - 4, 0), 8, 0.8, facecolor=resp_c[COMB.resp_k3_a[idx]], edgecolor="0.3", lw=.3))
a1.set_ylim(0, 2); a1.set_yticks([0.4, 1.4]); a1.set_yticklabels(["response", "region"])
a0.set_ylabel("Ward distance (normalized features)"); a0.set_title("combined clustering, first reading; region pink CeA, blue BLP; response black excited, grey inhibited, white none")
plt.show()
print("leaf order, top split: cluster of each leaf left to right:", "".join(str(COMB.cl[i]) for i in order))

**Result: not reproduced.** In every reading the combined clustering puts the CeA late-firing cells with the BLP
putative pyramidal cells (cluster 2) and the CeA regular-firing cells with the BLP putative interneurons (cluster 1);
the combined clusters agree with the separate ones in 51 of 52 cells. The CeA cells of cluster 2 are mostly excited
(11 to 13 of 17 or 19) and 5 or 6 of the 8 or 10 CeA cells of cluster 1 are inhibited, while the BLP cells of cluster 1
are mostly excited (6 of 8) and those of cluster 2 mostly inhibited (10 or 11 of 17): the excited cluster is not the
same in the two regions, as the criterion requires. This is the pattern Figure 6 supplement 1N draws, and the variant
that checks it holds in every reading at 2 and 3 SD (not at 5 SD, where 9 of the 17 BLP cells of cluster 2 are
excited).

## fig7-s1ef-red-vs-blue

> Ex vivo, red light evoked ChrimsonR-mediated potentials in CeA cells with much larger peak
> amplitude and area than the blue light (0.3 or 0.2 mW) used for imaging.

The ChrimsonR cells' sweeps are named by description: "635nm ... every 30s" (red alone), "...,
+0.3mW blue light" and "..., +0.2mW blue light" (read as the blue light conditions of Figure 7
supplement 1E-F), and "..., +0.3mW cont blue light" (red during constant blue, panels G-I). The
blue descriptions begin with the red protocol and do not say whether the blue sweeps also carried
red pulses; reading them as blue alone is an assumption, supported by their small responses and by
their onset, the same as that of the red sweeps (data check). Amplitude readings: (a) the largest
absolute deflection from the 5 s baseline, (b) the EPSP peak, or the IPSP trough magnitude when
only an IPSP is detected (2 or 3 SD). Area readings: the magnitude of the signed 0-5.5 s area, and
the area of the absolute deflection. Repeated measures one-way ANOVA across the three lights on the
cells with all three.

In [ ]:
def psp(avg, k):
    e = de.evoked(avg, k)
    a_abs = max(e["peak_mV"], -e["trough_mV"])
    if e["epsp"]:
        a_b = e["peak_mV"]
    elif e["ipsp"]:
        a_b = -e["trough_mV"]
    else:
        a_b = 0.0
    return {"amp_a": a_abs, f"amp_b_k{k}": a_b, "area_signed": abs(e["area_mVs"]), "area_abs": e["abs_area_mVs"],
            f"detected_k{k}": e["epsp"] or e["ipsp"]}
crow = []
for c in CELLS:
    if "Chrimson" not in c["virus"]:
        continue
    for cond in ("red", "blue03", "blue02", "red_contblue"):
        if cond in c["sweeps"]:
            avg = de.average(c["sweeps"][cond])
            r = {"path": c["path"], "subject": c["subject"], "sex": c["sex"], "cell": c["cell_id"], "cond": cond,
                 "n_sweeps": len(c["sweeps"][cond])}
            for k in KS:
                r.update(psp(avg, k))
            crow.append(r)
CHR = pd.DataFrame(crow)
print("ChrimsonR cells per condition:", CHR.groupby("cond").path.nunique().to_dict(),
      "; cells of mice labelled female (kept):", CHR[CHR.sex != "M"].groupby("cond").cell.unique().map(list).to_dict())

def ef_table(CH):
    three = CH[CH.cond.isin(["red", "blue03", "blue02"])]
    complete = three.groupby("path").cond.nunique()
    T3 = three[three.path.isin(complete[complete == 3].index)]
    EF = []
    for m in ["amp_a", "amp_b_k2", "amp_b_k3", "area_signed", "area_abs"]:
        res = AnovaRM(T3, depvar=m, subject="path", within=["cond"]).fit().anova_table
        w = T3.pivot(index="path", columns="cond", values=m)
        EF.append({"measure": m, "F": res["F Value"].iloc[0], "df": f"{res['Num DF'].iloc[0]:.0f},{res['Den DF'].iloc[0]:.0f}",
                   "p": res["Pr > F"].iloc[0], **{f"mean_{c}": w[c].mean() for c in ("red", "blue03", "blue02")},
                   "red>0.3mW": int((w.red > w.blue03).sum()), "red>0.2mW": int((w.red > w.blue02).sum()), "n": len(w),
                   "p_red_vs_0.3 (paired t)": stats.ttest_rel(w.red, w.blue03).pvalue,
                   "p_red_vs_0.2 (paired t)": stats.ttest_rel(w.red, w.blue02).pvalue})
    EF = pd.DataFrame(EF)
    EF["holds"] = (EF.p < 0.05) & (EF["red>0.3mW"] > EF.n / 2) & (EF["red>0.2mW"] > EF.n / 2)
    return EF, T3
EF, T3 = ef_table(CHR)
print("cells with all three lights:", T3.path.nunique(), "(paper: 9, from F(2,16))")
print(EF.round(4).to_string())
print("\nper cell (amplitude reading a, mV):")
print(T3.pivot(index=["subject", "cell"], columns="cond", values="amp_a").round(2))

**Result: reproduced, reading the blue sweeps as blue light alone.** In the 9 ChrimsonR cells with all three lights
(the paper's 9), red light evokes larger potentials than either blue light under every reading: amplitude F(2,16) =
25.6, p < 0.0001 (mean 9.9 mV against 1.8 and 0.9 mV), larger in 8 of 9 cells against each blue; area F(2,16) = 11.0
or 7.0, p = 0.001 or 0.0067, larger in 9 of 9. The paper's F values (200.1 and 404.2) are much larger; one cell
(C1, 0.68 mV to red) barely responds. The per-cell values support reading the blue sweeps as blue light alone: in
C4 and C9 of subject 235, red light during constant blue gives 16.1 and 11.9 mV (fig7-s1hi), while the blue sweeps
give 1.14 mV or less, so those sweeps cannot be red pulses under blue light.

## fig7-s1hi-red-during-blue

> Red light still evoked PSPs during constant blue light, though they were significantly smaller than
> with red light alone.

Red alone against red during constant 0.3 mW blue light ("+0.3mW cont blue light"), the same
measures, paired two-sided t-tests on the cells with both.

In [ ]:
def hi_table(CH):
    two = CH[CH.cond.isin(["red", "red_contblue"])]
    both = two.groupby("path").cond.nunique()
    T2 = two[two.path.isin(both[both == 2].index)]
    HI = []
    for m in ["amp_a", "amp_b_k2", "amp_b_k3", "area_signed", "area_abs"]:
        w = T2.pivot(index="path", columns="cond", values=m)
        t, p = stats.ttest_rel(w.red, w.red_contblue)
        HI.append({"measure": m, "t": t, "df": len(w) - 1, "p": p, "mean_red": w.red.mean(), "mean_red_during_blue": w.red_contblue.mean(),
                   "smaller_during_blue": int((w.red_contblue < w.red).sum()), "n": len(w)})
    HI = pd.DataFrame(HI)
    HI["holds"] = (HI.p < 0.05) & (HI.mean_red_during_blue < HI.mean_red)
    return HI, T2
HI, T2 = hi_table(CHR)
print("cells with both:", T2.path.nunique(), "(paper: 12, from t11)")
det = T2[T2.cond == "red_contblue"]
for k in KS:
    print(f"PSP detected during blue light at {k} SD: {int(det[f'detected_k{k}'].sum())} of {len(det)} cells; "
          f"with red alone: {int(T2[T2.cond == 'red'][f'detected_k{k}'].sum())} of {len(det)}")
print(HI.round(4).to_string())
print("\nper cell (amplitude reading a, mV):")
print(T2.pivot(index=["subject", "cell"], columns="cond", values="amp_a").round(2))

**Result: partially reproduced.** A PSP is detected during constant blue light in all 13 cells at 2 and 3 SD (10 of
13 at 5 SD). The amplitude is smaller during blue light (mean 4.1 against 8.1 mV, paired t(12) = 3.23, p = 0.0072,
smaller in 11 of 13; reading b p = 0.0083, 10 of 13), but the area is not significantly smaller (p = 0.26 and 0.16,
smaller in 12 and 11 of 13). The paper finds both smaller. Detection in 13 of 13 at 2 and 3 SD says little, since
the rule detects deflections in many light-free windows (fig6df); at 5 SD it is 10 of 13.

## In vivo imaging claims (blocked)

fig7e, fig7gh, fig7j, fig7i, fig7m, fig7-s1no, fig7-s1l and fig7-s2 all need, per session, when the
mouse interacted with the social and the object cup, or its time on each side (Methods: SLEAP pose
with 15 keypoints, the nose within 1.3 cup diameters and the cup in a 90 degree cone). The imaging
files hold the CNMF-E traces and footprints, the raw movie and a link to the behaviour video, and
nothing else: no pose, no interaction or position series, no epochs for the habituation and test
periods, and no record of which cup held the juvenile. The SLEAP model is not published. The
behaviour videos (data check) show the mouse and both cups from above, so side times could in
principle be tracked from them, but the cups are covered and nothing in the files says which held
the juvenile; tracking was not attempted. The footprints would allow the co-registration behind
fig7i's 202 cells, but its finding is the overlap of social-excited cells, which needs the
interactions; it was not run. The data check above shows what is there, and how the ROI counts
compare with the paper's cell counts.

## Variant: male mice only

The Methods use male mice only, and the files label the subjects of 17 ChR2 cells (subjects 23, C137 and C150)
female. The analysis above keeps them, because without them the dataset has fewer cells with current steps than the
paper clusters (data check). Here every tested claim is rerun on the cells of the mice labelled male, so that the
effect of that choice can be seen.

In [ ]:
# every claim tested above, rerun on the cells of the mice labelled male only 
RA = REST[REST.sex == "M"].reset_index(drop=True)
FA = FIRE[FIRE.sex == "M"].reset_index(drop=True)
FTA = FEAT[FEAT.sex == "M"].reset_index(drop=True)
CHA = CHR[CHR.sex == "M"].reset_index(drop=True)
cr, _ = cea_runs(FTA); br, _ = blp_runs(FTA); ln, _ = ln_table(FTA); jka = jk_table(FTA)
ja, ka = j_table(RA), k_table(RA)
efa, t3a = ef_table(CHA); hia, t2a = hi_table(CHA)
var = pd.DataFrame([
    {"claim": "fig6df", "cells": str({r: int((RA.region == r).sum()) for r in REGIONS}),
     "result": "; ".join(f"k={k}: {fig6df(RA, k)[1]}" for k in KS)},
    {"claim": "fig6gi", "cells": str({r: int((FA.region == r).sum()) for r in REGIONS}),
     "result": "every BNST cell with an IEI excited and both classes in CeA and BLP, all readings: "
               + str(bool(fig6gi(FA, "a")[["every BNST cell with an IEI excited", "both classes in CeA and BLP"]].all().all()))},
    {"claim": "fig6j", "cells": "", "result": "; ".join(f"k={r.k} ({r.cells}): p={r.p:.4f}, Dunn {r.dunn_CeA_BLP:.4f}, holds {r.holds}" for r in ja.itertuples())},
    {"claim": "fig6k", "cells": "", "result": "; ".join(f"k={r.k} ({r.cells}, n={r.n_total}): p={r.p:.4f}, holds {r.holds_tukey}/{r.holds_holm_sidak}" for r in ka.itertuples())},
    {"claim": "fig6no", "cells": str(cr.n.unique()), "result": "; ".join(f"k={k}: late excited {cr[cr.k == k].late_excited.min()}-{cr[cr.k == k].late_excited.max()} of {cr[cr.k == k].n_late.min()}-{cr[cr.k == k].n_late.max()}, holds {cr[cr.k == k].holds.any()}" for k in KS)},
    {"claim": "fig6-s1h", "cells": "", "result": "all p < 0.05 in every reading: " + str(bool(h_table(FTA)["all_p<0.05"].all()))},
    {"claim": "fig6pq", "cells": str(br.n.unique()), "result": "; ".join(f"k={k}: pyr inhibited {br[br.k == k]['pyr_inhibited_%'].min():.1f}-{br[br.k == k]['pyr_inhibited_%'].max():.1f}%, int excited {br[br.k == k]['int_excited_%'].min():.1f}-{br[br.k == k]['int_excited_%'].max():.1f}%, direction {br[br.k == k].direction.all()}" for k in KS)},
    {"claim": "fig6-s1i", "cells": "", "result": "all p < 0.05 in every reading: " + str(bool(i_table(FTA)["all_p<0.05"].all()))},
    {"claim": "fig6-s1jk", "cells": "", "result": "; ".join(f"k={k}: {jk_holds(jka, k)}" for k in KS)},
    {"claim": "fig6-s1ln", "cells": str(ln.n.unique()), "result": "; ".join(f"k={k}: criterion holds in {ln[ln.k == k].holds.sum()} of {(ln.k == k).sum()} readings, Figure 6-S1N pattern in {ln[ln.k == k].figure_s1n_pattern.sum()}" for k in KS)},
    {"claim": "fig7-s1ef", "cells": str(t3a.path.nunique()), "result": "; ".join(f"{r.measure}: p={r.p:.4f}, holds {r.holds}" for r in efa.itertuples())},
    {"claim": "fig7-s1hi", "cells": str(t2a.path.nunique()), "result": "; ".join(f"{r.measure}: p={r.p:.4f}, holds {r.holds}" for r in hia.itertuples())},
])
for r in var.itertuples():
    print(f"{r.claim}  {r.cells}\n    {r.result}")
print("\nfig6-s1jk with male mice only, every test and reading:")
print(jka.round(4).to_string())
print("\nfig6-s1ln with male mice only, every reading:")
print(ln[[c for c in ln.columns if not c.endswith("frac")]].to_string())

With the cells of the mice labelled male only, the outcomes of fig6df, fig6gi, fig6-s1h, fig6-s1i, fig7-s1ef and
fig7-s1hi do not change. fig6j holds in reading (a) at 2 and 3 SD (p = 0.022 and 0.014), and fig6k at 3 SD (p =
0.0052). The clustering claims lose cells (23 CeA and 22 BLP, against the paper's 26 and 27): fig6no still fails.
fig6-s1jk no longer holds: the BLP area (p = 0.11) and BLP EPSP peak (p = 0.099) tests are not significant, while
the CeA area test is (p = 0.0047 or below) and the CeA EPSP test is not (p = 0.076 to 0.32), as the criterion asks.
fig6pq keeps its direction at 2 and 3 SD (56 to 63% of the pyramidal cluster inhibited, 83% of the other excited)
and loses it at 5 SD (44%). In the combined clustering the criterion holds in no reading, and the pattern of Figure
6 supplement 1N holds in 6 of 8 readings at 2 SD, 4 of 8 at 3 SD and none at 5 SD. It fails mostly in the readings
with the ramp ratio from deflections, where the male-only clustering splits the cells differently and CeA cluster 2
is not clearly inhibited (4 to 6 excited and 6 to 8 inhibited of 14).


## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig6df-epsp-ipsp-at-rest | reproduced (cell counts differ: 15, 38, 56 against 19, 36, 48) |
| fig6gi-firing-responses | partially reproduced (3, 13, 13 firing cells against 5, 20, 17; one BNST cell cannot be classed) |
| fig6j-epsp-amplitude-region | partially reproduced (significant in one of the criterion's four readings, and at 5 SD) |
| fig6k-ipsp-amplitude-region | partially reproduced under the adopted 3 SD reading (58 cells against 34) |
| fig6-s1de-monosynaptic | blocked: TTX/4AP sweeps not marked |
| fig6no-cea-clusters | partially reproduced (2 or 3 inhibited cells in the late-firing cluster); not reproduced is proposed |
| fig6-s1h-cea-cluster-properties | reproduced |
| fig6pq-blp-clusters | partially reproduced (the direction fails at 5 SD) |
| fig6-s1i-blp-cluster-properties | reproduced (the delay test fails when one of 4 single cells is left out) |
| fig6-s1jk-cluster-voltage-area | reproduced, keeping the cells of the mice labelled female (the BLP tests fail without them) |
| fig6-s1ln-combined-clusters | not reproduced as the criterion asks; the pattern of Figure 6 supplement 1N holds (asked of a reader) |
| fig7-s1ef-red-vs-blue | reproduced, reading the blue sweeps as blue light alone |
| fig7-s1hi-red-during-blue | partially reproduced |
| In vivo imaging (8 claims) | blocked: no pose, interactions or cup sides |
| Figures 1 to 5 and 8 (17 claims) | blocked: data not in the dataset |

How the analysis differs from the paper's:

- The light time is not in the files; the first pulse is taken 5.468 s into each 30 s light sweep, where the
  responses start. Cells recorded only with other light protocols (20 s and 60 s sweeps, light at other times) are
  left out, and the counts of cells at rest and firing differ from the paper's.
- Rest sweeps that fire before the light are left out of the rest averages, as the Methods hold cells at rest below
  threshold, and 51 failed sweeps of one cell are left out.
- The paper's EPSP and IPSP were measured by hand in Clampfit with no stated threshold; here a deflection is
  detected beyond 2 or 3 SD of the 5 s baseline of the average trace, the criterion's readings. In a light-free window
  of the same traces this rule finds an EPSP in 94% and 80% of cells and an IPSP in 67% and 30%, so it finds more
  IPSPs than the paper reports; 5 SD (40% and 5%) is shown as a variant.
- The cells of the three subjects the files label female are kept, although the Methods use male mice; male mice
  only is shown as a variant.
- Capacitance is taken from the 5 mV seal-test pulses by the Q-method, not from the paper's 250 ms step and
  MATLAB code, which are not in the files or published.
- The ChrimsonR sweeps are matched to the light conditions by their descriptions.

What was found about the dataset:

- The light timing is not recorded; the stimulus series of the light sweeps are zero, and the protocol is given
  only in free text. The holding current the descriptions give disagrees with the measured current in 88 sweeps.
- In current clamp the injected current (ch-1) is labelled volts; the seal test is stored as an IZeroClampSeries but
  is voltage clamp. 51 sweeps of one cell (P10) are empty (0 mV) with a ch-1 outside any plausible current.
- No sweep or file is marked as recorded in TTX/4AP.
- Two cells named P (the BLP naming) are labelled BNST.
- Three subjects (17 ChR2 cells) are labelled female where the paper used male mice only, and some subject ids
  differ only by a suffix. Either the label or the paper is wrong, or these cells are not among the paper's.
- The imaging traces are stated at 25 Hz, but their length matches the behaviour video only at 20 Hz, the Methods'
  rate. The imaging files' virus, stimulus and experiment descriptions describe another experiment (GRIN lens in
  the mPFC, Chrimson in the BLA, 589 nm light at 20 Hz for 5 min).
- The imaging files hold about twice as many ROIs as the paper's cell counts (869, 913 and 854 against 429, 441 and
  484 for the ChrimsonR mice), with nothing marking the ROIs the paper kept, and no pose, interaction or cup-side
  data, so none of the in vivo findings can be tested.